In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import time
import os, shutil
import matplotlib.pyplot as plt
import tqdm
from torch.utils.tensorboard import SummaryWriter

In [ ]:


def dataset_aplit(query, train_cnt, val_cnt)  :
    origin_file = '../data/민원문서/'+query
    directory_list = [
        '../data/dataset/train/',
        '../data/dataset/val/',
        '../data/dataset/test/',
    ]

    for dir in directory_list:
        if not os.path.isdir(dir+query): #만약 파일이 없으면
            os.makedirs(dir+query)  #만들어라

    cnt = 0
    for file_name in os.listdir(origin_file):
        if cnt < train_cnt:
            print(f'trainset : {file_name}')
            shutil.move(origin_file+'/'+file_name, '../data/dataset/train/'+query+'/'+file_name)
        elif cnt < train_cnt + val_cnt:
            print(f'valset : {file_name}')
            shutil.move(origin_file+'/'+file_name, '../data/dataset/val/'+query+'/'+file_name)
        else:
            print(f'testset : {file_name}')
            shutil.move(origin_file+'/'+file_name, '../data/dataset/test/'+query+'/'+file_name)

        cnt += 1

    shutil.rmtree(origin_file)

#dataset_aplit('여권신청서',90,20)
#dataset_aplit('운전면허증',90,20)
#dataset_aplit('전입신고서',90,20)
dataset_aplit('주민등록증',90,20)
dataset_aplit('확정일자',90,20)

trainset : 주민등록증1.jpg
trainset : 주민등록증10.jpg
trainset : 주민등록증100.jpg
trainset : 주민등록증101.jpg
trainset : 주민등록증102.jpg
trainset : 주민등록증103.jpg
trainset : 주민등록증104.jpg
trainset : 주민등록증105.jpg
trainset : 주민등록증106.jpg
trainset : 주민등록증107.jpg
trainset : 주민등록증108.jpg
trainset : 주민등록증109.jpg
trainset : 주민등록증11.jpg
trainset : 주민등록증110.jpg
trainset : 주민등록증111.jpg
trainset : 주민등록증112.jpg
trainset : 주민등록증113.jpg
trainset : 주민등록증114.jpg
trainset : 주민등록증115.jpg
trainset : 주민등록증116.jpg
trainset : 주민등록증117.jpg
trainset : 주민등록증118.jpg
trainset : 주민등록증119.jpg
trainset : 주민등록증12.jpg
trainset : 주민등록증120.jpg
trainset : 주민등록증121.jpg
trainset : 주민등록증122.jpg
trainset : 주민등록증123.jpg
trainset : 주민등록증124.jpg
trainset : 주민등록증125.jpg
trainset : 주민등록증126.jpg
trainset : 주민등록증127.jpg
trainset : 주민등록증128.jpg
trainset : 주민등록증129.jpg
trainset : 주민등록증13.jpg
trainset : 주민등록증130.jpg
trainset : 주민등록증131.jpg
trainset : 주민등록증132.jpg
trainset : 주민등록증133.jpg
trainset : 주민등록증134.jpg
trainset : 주민등록증135.jpg
trainset : 주민등록증136.jp

In [12]:
transform_train = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]
)

transform_test = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]
)

In [13]:
train_datasets = datasets.ImageFolder(root='../data/dataset/train',transform=transform_train)
val_datasets = datasets.ImageFolder(root='../data/dataset/val',transform=transform_test)
test_datasets = datasets.ImageFolder(root='../data/dataset/test',transform=transform_test)

In [14]:
train_datasets.classes

['여권신청서', '운전면허증', '전입신고서', '주민등록증', '확정일자']

In [15]:
train_datasets[0]

(tensor([[[2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5536],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5364],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5364],
          ...,
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5364],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5364],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 0.5364]],
 
         [[2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6954],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6779],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6779],
          ...,
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6779],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6779],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 0.6779]],
 
         [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 0.9145],
          [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 0.8971],
          [2.6400, 2.6400, 2.6400,  ...,

In [19]:
def imshow(img,title):    
    mean = torch.tensor([0.485,0.456,0.406])
    std = torch.tensor([0.229,0.224,0.225])
    img = img.permute(1,2,0)
    img = img * std + mean
    plt.title(title)
    plt.imshow(img)
    plt.show()

In [30]:
train_loader = torch.utils.data.DataLoader(train_datasets, shuffle=True, batch_size=4)
val_loader = torch.utils.data.DataLoader(val_datasets, shuffle=True, batch_size=4)
test_loader = torch.utils.data.DataLoader(test_datasets, shuffle=True, batch_size=4)

In [35]:
model = models.efficientnet_b0(pretrained=True)

model


c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [43]:
for params in model.parameters():
    params.requires_grad = False

for params in model.features[8].parameters():
    params.requires_grad = True

for params in model.classifier.parameters():
    params.requires_grad = True

model.classifier[1] = nn.Linear(1280,5)

In [44]:
for name,module in model.named_parameters():
    print(name, module.requires_grad)

model


features.0.0.weight False
features.0.1.weight False
features.0.1.bias False
features.1.0.block.0.0.weight False
features.1.0.block.0.1.weight False
features.1.0.block.0.1.bias False
features.1.0.block.1.fc1.weight False
features.1.0.block.1.fc1.bias False
features.1.0.block.1.fc2.weight False
features.1.0.block.1.fc2.bias False
features.1.0.block.2.0.weight False
features.1.0.block.2.1.weight False
features.1.0.block.2.1.bias False
features.2.0.block.0.0.weight False
features.2.0.block.0.1.weight False
features.2.0.block.0.1.bias False
features.2.0.block.1.0.weight False
features.2.0.block.1.1.weight False
features.2.0.block.1.1.bias False
features.2.0.block.2.fc1.weight False
features.2.0.block.2.fc1.bias False
features.2.0.block.2.fc2.weight False
features.2.0.block.2.fc2.bias False
features.2.0.block.3.0.weight False
features.2.0.block.3.1.weight False
features.2.0.block.3.1.bias False
features.2.1.block.0.0.weight False
features.2.1.block.0.1.weight False
features.2.1.block.0.1.bia

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [45]:
writer = SummaryWriter()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = optim.Adam(model.parameters(), lr = 3e-5)
criterion = nn.CrossEntropyLoss()

model.to(device)

best_val_loss = 100000000
epochs = 30
early_stop_epochs = 5
early_stop_counter = 0
count = 0

for epoch in range(epochs):
    train_tqdm = tqdm.tqdm(train_loader)
    model.train()
    train_loss_sum = 0

    for img,labels in train_tqdm:
        optimizer.zero_grad()
        preds = model(img.to(device))
        loss = criterion(preds,labels.to(device))
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()
        writer.add_scalar("Loss/train_step", loss.item(), count)
        count += 1
        train_tqdm.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)
    print("avg_train_loss",avg_train_loss)

    model.eval()    
    all_preds = []
    all_labels = []
    val_loss_sum = 0
    with torch.no_grad():
        for img,labels in val_loader:
            img = img.to(device)
            labels = labels.to(device)
            pred = model(img)
            loss = criterion(pred,labels)
            val_loss_sum += loss.item()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

        avg_val_loss = val_loss_sum / len(val_loader)
    print( "avg_val_loss",avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "document_best_model.pt")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

        if early_stop_counter >= early_stop_epochs:
            print("Early stopping triggered.")
            break


100%|██████████| 113/113 [00:09<00:00, 12.47it/s, loss=1.6090]


avg_train_loss 1.4768792686209213
avg_val_loss 1.3336553812026977


100%|██████████| 113/113 [00:08<00:00, 13.03it/s, loss=1.4216]


avg_train_loss 1.1905406803156422
avg_val_loss 1.131742572784424


100%|██████████| 113/113 [00:08<00:00, 12.97it/s, loss=1.8397]


...

avg_train_loss 0.16601125547349188
avg_val_loss 0.09416858483105899


100%|██████████| 113/113 [00:09<00:00, 12.37it/s, loss=0.3516]


avg_train_loss 0.15692854469098086
avg_val_loss 0.0814419700205326


100%|██████████| 113/113 [00:08<00:00, 12.86it/s, loss=0.1083]


avg_train_loss 0.11117750603244105
avg_val_loss 0.06035092651844025
Early stopping triggered.


In [46]:
model.eval()
with torch.no_grad():
    corrects = 0

    for img,labels in test_loader:
        preds = model(img.to(device))
        pred = torch.max(preds,1)[1]

        corrects += torch.sum(pred ==labels.to(device).data)
        img_grid = torchvision.utils.make_grid(img)
        #imshow(img_grid.cpu(), title=(pred))

        print(labels)

    acc = corrects / len(test_datasets.targets)
    print ('정확도 : ', acc)

tensor([4, 3, 2, 2])
tensor([2, 2, 4, 3])
tensor([0, 4, 4, 0])
tensor([0, 0, 4, 1])
tensor([3, 0, 4, 3])
tensor([1, 4, 4, 4])
tensor([4, 3, 2, 1])
tensor([1, 2, 0, 4])
tensor([0, 3, 1, 1])
tensor([1, 0, 2, 0])
tensor([0, 1, 4, 3])
tensor([0, 4, 1, 4])
tensor([2, 1, 1, 2])
tensor([1, 3, 3, 4])
tensor([4, 1, 3, 2])
tensor([0, 2, 1, 0])
tensor([1, 1, 3, 4])
tensor([2, 2, 2, 0])
tensor([2, 2, 1, 4])
tensor([1, 4, 1, 4])
tensor([0, 1, 3, 1])
tensor([0, 4, 2, 0])
tensor([3, 1, 2, 0])
tensor([0, 1, 4, 2])
tensor([3, 1, 4, 0])
tensor([3, 3, 2, 0])
tensor([2, 3, 3, 2])
tensor([4, 0, 0, 4])
tensor([2, 2, 0, 3])
tensor([2, 0, 3, 3])
tensor([3])
정확도 :  tensor(1.0000, device='cuda:0')
